# Create MERlin scripts

Writes the MERlin config and run files to `SAMPLE_DIR/merlin/`. Codebooks and
microscope parameters are not copied: the slurm script points at them in this
clone (`MERci/data/configs/merlin/`).

Run after notebook 08 (needs `experiment_info.yaml`).

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/multi_z/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MERLIN_DIR   = SAMPLE_DIR / "merlin"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import load_experiment_info, positions_file_tag
from MERci.acquisition.display        import display_file
from MERci.acquisition.merlin_config import (
    resolve_codebook_filename, resolve_microscope_parameters_filename,
    build_merlin_analysis_parameters,
    create_cluster_resource_allocation, create_snakemake_parameters,
    resolve_cluster_sample_dir, short_experiment_name, create_slurm_submit_script,
)

info = load_experiment_info(METADATA_DIR / "experiment_info.yaml")
MICROSCOPE  = info.microscope
OBJECTIVE   = info.extra.get("objective")   # None -> the microscope's default

# SAMPLE_NAME from experiment_info.yaml names the files the other notebooks wrote.
SAMPLE_NAME = info.sample_name

# SHORT_NAME: SLURM-safe short name for the merlin/ files, e.g. merlin_slurm_s5-BC555e.sh
SHORT_NAME = short_experiment_name(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

# Positions files are positions_{POSITIONS_TAG}.txt (notebook 03).
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, info.extra.get("imaging_dir", ""))

print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"SHORT_NAME  : {SHORT_NAME}")
print(f"MICROSCOPE  : {MICROSCOPE}")
print(f"OBJECTIVE   : {OBJECTIVE}")
print(f"lib_name    : {info.lib_name}")
print(f"data_home   : {info.data_home}")
print(f"merlin_home : {info.merlin_home}")
print(f"folder_name : {info.folder_name}")

## Resolve shared reference files (codebook, microscope parameters)

From `MERci/data/configs/merlin/`. The codebook's bit count is checked against
`round_bit_color_map.csv`.

In [ ]:
codebook_path   = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / resolve_codebook_filename(info.lib_name)
microscope_path = MERCI_DIR / "data" / "configs" / "merlin" / "microscope" / resolve_microscope_parameters_filename(MICROSCOPE, OBJECTIVE)

for p in (codebook_path, microscope_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found -- add it to MERci/data/configs/merlin/ before running this notebook."
        )

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if rbc_path.exists():
    # Distinct bits, not round.max(): a round can image several bits.
    n_bits = int(pd.read_csv(rbc_path)["bit"].nunique())
    codebook_bit_names_line = next(
        line for line in codebook_path.read_text().splitlines() if line.startswith("bit_names")
    )
    codebook_n_bits = len(codebook_bit_names_line.split(",")) - 1
    if codebook_n_bits != n_bits:
        print(f"WARNING: codebook {codebook_path.name} has {codebook_n_bits} bits, "
              f"but this experiment images {n_bits} -- double-check lib_name/codebook choice.")
    else:
        print(f"Bit count OK: {n_bits} bits, matching {codebook_path.name}.")
else:
    n_bits = None
    print(f"WARNING: {rbc_path} not found -- run notebook 05 first.")

print(f"Codebook  : {codebook_path}")
print(f"Microscope: {microscope_path}")

## MERlin analysis-parameters JSON

Built from a recipe (`data/configs/merlin/analysis/recipes/`) of MERlin task
files (`data/configs/merlin/analysis/tasks/`). Edit those files to tune
parameters.

In [ ]:
# "default_with_segmentation.yaml" adds cell segmentation (CellPoseSegment3D;
# edit the recipe for CellPoseSegmentSAM).
ANALYSIS_RECIPES_DIR = MERCI_DIR / "data" / "configs" / "merlin" / "analysis" / "recipes"
ANALYSIS_TASKS_DIR   = MERCI_DIR / "data" / "configs" / "merlin" / "analysis" / "tasks"
N_OPTIMIZE_ITERATIONS = info.extra.get("n_opt", 10)   # also used in the snakemake cell

RECIPE_PATH = ANALYSIS_RECIPES_DIR / "default_no_segmentation.yaml"

analysis_path = MERLIN_DIR / "analysis" / f"merlin_analysis_{SHORT_NAME}.yaml"
build_merlin_analysis_parameters(
    recipe_path            = RECIPE_PATH,
    tasks_dir               = ANALYSIS_TASKS_DIR,
    output_path              = analysis_path,
    n_optimize_iterations     = N_OPTIMIZE_ITERATIONS,
)
print(f"Saved: {analysis_path}")
display_file(analysis_path)

## Snakemake cluster-resource-allocation + parameters YAML

In [ ]:
cluster_template = MERCI_DIR / "data" / "configs" / "merlin" / "snakemake" / "cluster_resource_allocation_basic.json"

cluster_resource_path = MERLIN_DIR / "snakemake" / f"cluster_resource_allocation_{SHORT_NAME}.yaml"
create_cluster_resource_allocation(
    template_path=cluster_template, exp_name=SAMPLE_NAME,
    n_optimize_iterations=N_OPTIMIZE_ITERATIONS, output_path=cluster_resource_path,
)
print(f"Saved: {cluster_resource_path}")
display_file(cluster_resource_path)

snakemake_params_path = MERLIN_DIR / "snakemake" / f"parameters_{SHORT_NAME}.yaml"
create_snakemake_parameters(
    exp_name=SAMPLE_NAME, cluster_config_path=cluster_resource_path,
    output_path=snakemake_params_path,
    job_name_prefix=SHORT_NAME,
)
print(f"Saved: {snakemake_params_path}")
display_file(snakemake_params_path)

## Slurm submit script

Every path in the script is relative to one `$SAMPLE_DIR` variable, so it is
the same whether made here before the transfer (path predicted from the
sample name) or on the cluster (real path). See `resolve_cluster_sample_dir`.

In [ ]:
data_org_path  = MERLIN_DIR / "dataorganization" / f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}.txt"

# Use the camera-rotation-corrected positions (misc/correct_camera_rotation)
# when they exist.
corrected_positions_path = POSITIONS_DIR / f"positions_{POSITIONS_TAG}_corrected.txt"
if corrected_positions_path.exists():
    positions_path = corrected_positions_path
    print(f"Using camera-rotation-corrected positions file: {positions_path.name} "
          f"(see notebooks/misc/correct_camera_rotation.ipynb)")

for p in (data_org_path, positions_path):
    if not p.exists():
        print(f"WARNING: {p} not found. For a multi-boundary experiment, set the "
              f"correct per-segment file path manually before submitting to the cluster.")

# The acquisition root on the cluster (predicted on Windows, real on Linux).
sample_dir_cluster = resolve_cluster_sample_dir(SAMPLE_DIR, SAMPLE_NAME, info.extra.get("imaging_dir", ""))

def _rel(p):
    """POSIX path relative to SAMPLE_DIR, for use as "$SAMPLE_DIR/<...>" in the script."""
    return Path(p).relative_to(SAMPLE_DIR).as_posix()

# Variable-z round: pass --allow-ragged-z-stacks (thin FOVs have fewer frames).
# Needs a MERlin with ragged-z-stack support.
round_info_path = METADATA_DIR / "round_info.csv"
ALLOW_RAGGED_Z_STACKS = False
if round_info_path.exists():
    _round_info = pd.read_csv(round_info_path)
    ALLOW_RAGGED_Z_STACKS = ("tissue_thickness" in _round_info.columns
                             and (_round_info["tissue_thickness"] == "multi").any())
print(f"ALLOW_RAGGED_Z_STACKS: {ALLOW_RAGGED_Z_STACKS}")

submit_path = MERLIN_DIR / "slurm" / "submit" / f"merlin_slurm_{SHORT_NAME}.sh"
create_slurm_submit_script(
    label                   = SHORT_NAME,
    sample_dir              = sample_dir_cluster,
    parameters_file         = _rel(snakemake_params_path),
    analysis_file           = _rel(analysis_path),
    data_organization_file  = _rel(data_org_path),
    positions_file          = _rel(positions_path),
    codebook_file           = _rel(codebook_path),
    microscope_file         = _rel(microscope_path),
    data_home               = info.data_home,
    folder_name             = info.folder_name,
    output_path             = submit_path,
    allow_ragged_z_stacks   = ALLOW_RAGGED_Z_STACKS,
    analysis_name           = "output",
)
print(f"$SAMPLE_DIR (cluster) : {sample_dir_cluster}")
print(f"Saved: {submit_path}\n")
display_file(submit_path)